# Flu Shot Learning - notebook 5

This version uses the same respondent-level features for three complementary models. Missingness and ordinal survey signals are made explicit, while CatBoost and LightGBM handle categorical interactions and logistic regression supplies a stable linear baseline. The final blend is selected only from out-of-fold predictions.

In [1]:
from pathlib import Path
import importlib.util
import sys

missing = []
if importlib.util.find_spec('catboost') is None:
    missing.append('catboost')
if importlib.util.find_spec('lightgbm') is None:
    missing.append('lightgbm')
if importlib.util.find_spec('iterstrat') is None:
    missing.append('iterative-stratification')
if missing:
    !"{sys.executable}" -m pip install -q {' '.join(missing)}

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
pd.set_option('display.max_columns', 100)
print('Ready')

Ready


In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        
    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the folder containing the competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
assert train_features['respondent_id'].is_unique
assert test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])
y = train_labels[TARGETS].astype('int8')
print(f'Using: {DATA_DIR.resolve()}')
print(f'Train: {train_features.shape}; test: {test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())

Using: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 36); test: (26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)
    behavior = [c for c in result if c.startswith('behavioral_')]
    opinions = [c for c in result if c.startswith('opinion_')]
    risk = [c for c in opinions if '_risk' in c]
    result['behavior_score'] = result[behavior].sum(axis=1, min_count=1)
    result['behavior_available'] = result[behavior].notna().sum(axis=1)
    result['opinion_score'] = result[opinions].mean(axis=1)
    result['opinion_risk_score'] = result[risk].mean(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)
    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        result[column + '_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_columns]
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for column in categorical_columns:
    levels = pd.concat([X_lgb[column], X_test_lgb[column]], ignore_index=True).unique()
    dtype = pd.api.types.CategoricalDtype(categories=levels)
    X_lgb[column] = X_lgb[column].astype(dtype)
    X_test_lgb[column] = X_test_lgb[column].astype(dtype)
stratify_key = y[TARGETS].astype(str).agg('_'.join, axis=1)
folds = list(MultilabelStratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
).split(X, y))
print(f'Engineered features: {X.shape[1]}; categorical: {len(categorical_columns)}')

Engineered features: 46; categorical: 12


In [4]:
def make_catboost(target, seed):
    settings = {
        'h1n1_vaccine': {'depth': 7, 'iterations': 1900, 'learning_rate': 0.035, 'l2_leaf_reg': 7.0},
        'seasonal_vaccine': {'depth': 6, 'iterations': 1900, 'learning_rate': 0.035, 'l2_leaf_reg': 8.0},
    }[target]
    return CatBoostClassifier(
        **settings, loss_function='Logloss', eval_metric='AUC', random_seed=seed,
        verbose=False, allow_writing_files=False, thread_count=-1,
    )

def make_lightgbm(seed):
    return LGBMClassifier(
        objective='binary', n_estimators=2400, learning_rate=0.025,
        num_leaves=31, min_child_samples=30, subsample=0.85,
        colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, verbosity=-1,
    )

numeric_columns = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
linear_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler(with_mean=False)),
    ]), numeric_columns),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ]), categorical_columns),
], sparse_threshold=1.0)

def make_linear(target):
    return Pipeline([
        ('preprocessor', linear_preprocessor),
        ('classifier', LogisticRegression(
            C=0.35 if target == 'h1n1_vaccine' else 0.20,
            solver='saga', penalty='l2', max_iter=2500,
            random_state=RANDOM_STATE, n_jobs=-1,
        )),
    ])

In [5]:
oof = {name: pd.DataFrame(index=X.index, columns=TARGETS, dtype=float) for name in ['catboost', 'lightgbm', 'linear']}
test_pred = {name: pd.DataFrame(0.0, index=X_test.index, columns=TARGETS) for name in oof}

for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
        print(f'Fold {fold_number}/{N_SPLITS}')
        for target in TARGETS:
            cat = make_catboost(target, RANDOM_STATE + fold_number)
            cat.fit(X.iloc[train_index], y.iloc[train_index][target],
                    cat_features=cat_feature_indices,
                    eval_set=(X.iloc[valid_index], y.iloc[valid_index][target]),
                    use_best_model=True, early_stopping_rounds=140)
            oof['catboost'].loc[valid_index, target] = cat.predict_proba(X.iloc[valid_index])[:, 1]
            test_pred['catboost'][target] += cat.predict_proba(X_test)[:, 1] / N_SPLITS

            lgb = make_lightgbm(RANDOM_STATE + 100 + fold_number)
            lgb.fit(X_lgb.iloc[train_index], y.iloc[train_index][target],
                    eval_set=[(X_lgb.iloc[valid_index], y.iloc[valid_index][target])],
                    eval_metric='auc', categorical_feature=categorical_columns,
                    callbacks=[early_stopping(140, verbose=False), log_evaluation(0)])
            oof['lightgbm'].loc[valid_index, target] = lgb.predict_proba(X_lgb.iloc[valid_index])[:, 1]
            test_pred['lightgbm'][target] += lgb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

            linear = make_linear(target)
            linear.fit(X.iloc[train_index], y.iloc[train_index][target])
            oof['linear'].loc[valid_index, target] = linear.predict_proba(X.iloc[valid_index])[:, 1]
            test_pred['linear'][target] += linear.predict_proba(X_test)[:, 1] / N_SPLITS

model_scores = pd.DataFrame({name: [roc_auc_score(y[target], oof[name][target]) for target in TARGETS] for name in oof}, index=TARGETS)
model_scores.loc['mean'] = model_scores.mean(axis=0)
display(model_scores)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5


,catboost,lightgbm,linear
h1n1_vaccine,0.871272,0.870635,0.861974
seasonal_vaccine,0.864278,0.863869,0.854972
mean,0.867775,0.867252,0.858473


In [6]:
def blend_score(weights, predictions):
    scores = []

    for target in TARGETS:
        blended = sum(
            weights[name] * predictions[name][target]
            for name in predictions
        )
        scores.append(roc_auc_score(y[target], blended))

    return float(np.mean(scores)), scores


candidates = []

for cat_weight in np.arange(0.0, 1.01, 0.1):
    for lgb_weight in np.arange(0.0, 1.01 - cat_weight, 0.1):

        linear_weight = round(1.0 - cat_weight - lgb_weight, 10)

        weights = {
            'catboost': float(cat_weight),
            'lightgbm': float(lgb_weight),
            'linear': linear_weight
        }

        score, target_scores = blend_score(weights, oof)

        candidates.append(
            (score, weights, target_scores)
        )


best_score, best_weights, best_target_scores = max(
    candidates,
    key=lambda item: item[0]
)

print('Selected OOF weights:', best_weights)
print(
    'Target AUC:',
    dict(zip(TARGETS, np.round(best_target_scores, 6)))
)
print(f'Mean OOF ROC AUC: {best_score:.6f}')

Selected OOF weights: {'catboost': 0.5, 'lightgbm': 0.4, 'linear': np.float64(0.1)}
Target AUC: {'h1n1_vaccine': np.float64(0.872386), 'seasonal_vaccine': np.float64(0.865232)}
Mean OOF ROC AUC: 0.868809


In [7]:
final_predictions = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
for target in TARGETS:
        final_predictions[target] = sum(best_weights[name] * test_pred[name][target] for name in test_pred)
submission = pd.DataFrame({
        'respondent_id': test_features['respondent_id'],
        'h1n1_vaccine': np.clip(final_predictions['h1n1_vaccine'], 0.0, 1.0),
        'seasonal_vaccine': np.clip(final_predictions['seasonal_vaccine'], 0.0, 1.0),
    })
assert list(submission.columns) == ['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()

output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission_notebook5_blended.csv'
oof_path = output_dir / 'oof_notebook5_blended.csv'
submission.to_csv(submission_path, index=False)
oof_frame = oof['catboost'].rename(columns=lambda c: f'catboost_{c}')
for name in ['lightgbm', 'linear']:
        oof_frame = oof_frame.join(oof[name].rename(columns=lambda c, n=name: f'{n}_{c}'))
oof_frame.insert(0, 'respondent_id', train_features['respondent_id'])
oof_frame.to_csv(oof_path, index=False)
print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())
display(submission[TARGETS].describe().loc[['min', 'mean', 'max']])

Saved submission: /kaggle/working/outputs/submission_notebook5_blended.csv
Saved OOF predictions: /kaggle/working/outputs/oof_notebook5_blended.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.111384,0.219698
1,26708,0.031334,0.036262
2,26709,0.184499,0.649800
3,26710,0.641652,0.888779
4,26711,0.277405,0.456748


,h1n1_vaccine,seasonal_vaccine
min,0.005065,0.011419
mean,0.208928,0.465187
max,0.974404,0.987143
